In [1]:
%pip install -q sagemaker boto3 pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


# Week 2 Tuesday -- Advanced SageMaker Data Engineering

On Monday you deployed your trained models, invoked them with held-out data, and computed performance metrics. You also learned MLOps concepts -- pipelines, DAGs, and Model Registry. But the models consumed raw CSVs with no feature management, no consistency guarantees, and no reuse across teams.

Today you solve that problem. By the end of this notebook you will have:

1. **Created a Feature Group** in SageMaker Feature Store with fraud detection features, ingested data, and queried both online and offline stores.
2. **Explored Data Wrangler, Canvas, and Autopilot** -- understanding when each tool is appropriate and how they compare.
3. **Connected Feature Store to Pipelines** -- building a pipeline that reads from Feature Store, trains a model, and registers it in Model Registry.

| Block | Content | Minutes |
|-------|---------|--------|
| Stage 1 | Feature Store Fundamentals | 55 |
| Break 1 | Stretch / Questions | 5 |
| Stage 2 | Data Wrangler, Canvas, and Autopilot | 55 |
| Break 2 | Stretch / Questions | 5 |
| Stage 3 | Exporting to Pipelines | 55 |

## Setup

Run the cell below to establish your SageMaker session. This connects to your execution role, default S3 bucket, and region.

In [2]:
import boto3
import sagemaker
import time
import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = session.boto_region_name
bucket = session.default_bucket()
prefix = "fraudshield"

sm_client = boto3.client("sagemaker")
s3 = boto3.client("s3")
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region:   {region}")
print(f"Account:  {account_id}")
print(f"Bucket:   s3://{bucket}")
print(f"Role:     ...{role[-30:]}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region:   us-west-2
Account:  407975137156
Bucket:   s3://sagemaker-us-west-2-407975137156
Role:     ...-ExecutionRole-20260327T083133


---
# STAGE 1 -- Feature Store Fundamentals

**Connecting to Monday:** On Monday you deployed models that consumed raw CSVs. The features were computed in the notebook and passed directly to the endpoint. In production, this approach breaks down: training computes features one way, inference computes them another way, and the model silently degrades. This is **training-serving skew**.

SageMaker Feature Store eliminates this problem by providing a centralized, versioned store that serves both training (offline store) and real-time inference (online store) from the same feature definitions.

## Feature Store Concepts

### What is a Feature Group?

A **Feature Group** is a logical collection of features for a specific entity (e.g., a transaction, a customer). Think of it as a managed table with two access patterns:

- **Online Store** -- Low-latency key-value lookup (backed by DynamoDB). You query by record ID and get the latest feature values in single-digit milliseconds. Used for real-time inference.
- **Offline Store** -- Full historical data stored as Parquet files in S3, queryable via Athena. Used for building training datasets.

### Why Does This Matter for FraudShield?

When a transaction arrives at FraudShield in production:
1. The application calls the online store to get the latest features for that record
2. Those features are sent to the inference endpoint
3. The endpoint returns a fraud/not-fraud prediction

When you retrain the model:
1. A pipeline step queries the offline store via Athena for historical features
2. The same feature definitions, same computation, same schema
3. No training-serving skew

### Required Special Columns

Every Feature Group requires:
- **Record identifier** -- unique key for each entity (e.g., `record_id`)
- **Event time** -- timestamp for when the feature values were generated (enables point-in-time queries)

> **Discussion:** Think about FraudShield. If the fraud model uses `merchant_risk_score` during training but the production system computes it with a different formula, what happens to fraud detection accuracy?

## STEP 1 -- Generate Synthetic Fraud Data

We generate the same synthetic fraud dataset used throughout the week, adding the `record_id` and `event_time` columns required by Feature Store.

In [ ]:
from datetime import datetime, timedelta

np.random.seed(42)
n = 2000

fraud_data = pd.DataFrame({
    "record_id": [f"txn-{i:05d}" for i in range(n)],
    "amount": np.random.exponential(500, n).round(2),
    "hour": np.random.randint(0, 24, n),
    "distance_from_home": np.random.exponential(50, n).round(2),
    "transaction_count_24h": np.random.poisson(5, n),
    "is_international": np.random.binomial(1, 0.1, n),
    "merchant_risk_score": np.random.uniform(0, 1, n).round(3),
})

fraud_data["target"] = (
    (fraud_data["amount"] > 800) & (fraud_data["hour"] < 6)
    | (fraud_data["merchant_risk_score"] > 0.85)
).astype(int)
noise = np.random.random(n) < 0.08
fraud_data["target"] = (fraud_data["target"] ^ noise.astype(int))

base_time = datetime.utcnow()
fraud_data["event_time"] = [
    (base_time - timedelta(minutes=n - i)).strftime("%Y-%m-%dT%H:%M:%SZ")
    for i in range(n)
]

print(f"Dataset shape: {fraud_data.shape}")
print(f"Fraud rate: {fraud_data['target'].mean():.2%}")
print(f"\nSchema:")
print(fraud_data.dtypes)
print(f"\nSample rows:")
fraud_data.head()

Dataset shape: (2000, 9)
Fraud rate: 24.25%

Schema:
record_id                 object
amount                   float64
hour                       int64
distance_from_home       float64
transaction_count_24h      int64
is_international           int64
merchant_risk_score      float64
target                     int64
event_time                object
dtype: object

Sample rows:


/tmp/ipykernel_406/2890221094.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time = datetime.utcnow()


,record_id,amount,hour,distance_from_home,transaction_count_24h,is_international,merchant_risk_score,target,event_time
0,txn-00000,234.63,6,54.31,7,0,0.246,0,2026-03-30T05:50:00Z
1,txn-00001,1505.06,2,21.31,6,0,0.511,1,2026-03-30T05:51:00Z
2,txn-00002,658.37,22,24.78,8,0,0.385,0,2026-03-30T05:52:00Z
3,txn-00003,456.47,13,8.64,2,0,0.211,0,2026-03-30T05:53:00Z
4,txn-00004,84.81,23,14.94,3,0,0.751,0,2026-03-30T05:54:00Z


## STEP 2 -- Create a Feature Group

We define a Feature Group called `fraudshield-transaction-features` with the schema for our fraud detection features. The Feature Group is configured with both online and offline stores enabled -- the dual-access pattern.

**What is happening below:**
1. We define the feature definitions (column names and types)
2. We create the Feature Group with `enable_online_store=True` and an S3 URI for the offline store
3. We wait until the Feature Group status reaches `Created`

> **Discussion:** Why do we enable both online and offline stores? What would happen if we only enabled one?

In [4]:
from sagemaker.feature_store.feature_definition import FeatureDefinition, FeatureTypeEnum
from sagemaker.feature_store.feature_group import FeatureGroup

FEATURE_GROUP_NAME = "fraudshield-transaction-features"

feature_group = FeatureGroup(
    name=FEATURE_GROUP_NAME,
    sagemaker_session=session,
)

feature_definitions = [
    FeatureDefinition(feature_name="record_id", feature_type=FeatureTypeEnum.STRING),
    FeatureDefinition(feature_name="amount", feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="hour", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="distance_from_home", feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="transaction_count_24h", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="is_international", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="merchant_risk_score", feature_type=FeatureTypeEnum.FRACTIONAL),
    FeatureDefinition(feature_name="target", feature_type=FeatureTypeEnum.INTEGRAL),
    FeatureDefinition(feature_name="event_time", feature_type=FeatureTypeEnum.STRING),
]

feature_group.feature_definitions = feature_definitions

feature_group.create(
    s3_uri=f"s3://{bucket}/{prefix}/feature-store/",
    record_identifier_name="record_id",
    event_time_feature_name="event_time",
    role_arn=role,
    enable_online_store=True,
)

print(f"Creating Feature Group: {FEATURE_GROUP_NAME}")
print("Waiting for Feature Group to become active...")

status = "Creating"
while status == "Creating":
    time.sleep(5)
    response = sm_client.describe_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
    status = response["FeatureGroupStatus"]
    print(f"  Status: {status}")

if status == "Created":
    print(f"\nFeature Group '{FEATURE_GROUP_NAME}' is ready.")
else:
    print(f"\nUnexpected status: {status}. Check the AWS console for details.")

Creating Feature Group: fraudshield-transaction-features
Waiting for Feature Group to become active...
  Status: Creating
  Status: Creating
  Status: Creating
  Status: Created

Feature Group 'fraudshield-transaction-features' is ready.


## STEP 3 -- Ingest Data into the Feature Group

Now we load the fraud data into the Feature Group using bulk ingestion. The `ingest` method sends records from a DataFrame into the Feature Group. Each record is written to both the online store (for real-time lookups) and the offline store (for historical queries).

**Key details:**
- Ingestion is **idempotent** -- re-ingesting the same `record_id` with a newer `event_time` overwrites the online store value
- The offline store **appends** rather than overwrites, preserving full history
- Bulk ingestion runs asynchronously using multiple workers

> **Discussion:** Why does the online store overwrite while the offline store appends? Think about what each store is optimized for.

In [5]:
ingestion_manager = feature_group.ingest(
    data_frame=fraud_data,
    max_workers=4,
    wait=True,
)

failed_count = len(ingestion_manager.failed_rows)
print(f"Ingestion complete.")
print(f"  Total records: {len(fraud_data)}")
print(f"  Failed records: {failed_count}")

if failed_count > 0:
    print(f"  Failed row indices: {ingestion_manager.failed_rows}")
else:
    print("  All records ingested successfully.")

print("\nWaiting 30 seconds for online store propagation...")
time.sleep(30)
print("Ready for online store queries.")

Ingestion complete.
  Total records: 2000
  Failed records: 0
  All records ingested successfully.

Waiting 30 seconds for online store propagation...
Ready for online store queries.


## STEP 4 -- Query the Online Store

The online store provides **low-latency** feature retrieval by record ID. In production, this is how your inference endpoint gets features: a transaction arrives, you look up the record by ID, and you send the feature vector to the model.

**What is happening below:** We use the Feature Store runtime client to call `get_record` with a known record ID. The response contains the latest feature values for that record.

> **Discussion:** How does this compare to the approach on Monday, where we loaded an entire CSV and sent rows to the endpoint? Which approach scales better for real-time inference?

In [6]:
featurestore_runtime = boto3.client("sagemaker-featurestore-runtime")

test_record_ids = ["txn-00000", "txn-00050", "txn-00500"]

for record_id in test_record_ids:
    response = featurestore_runtime.get_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierValueAsString=record_id,
    )

    print(f"\n--- Record: {record_id} ---")
    for feature in response["Record"]:
        print(f"  {feature['FeatureName']:>25s}: {feature['ValueAsString']}")


--- Record: txn-00000 ---
                  record_id: txn-00000
                     amount: 234.63
                       hour: 6
         distance_from_home: 54.31
      transaction_count_24h: 7
           is_international: 0
        merchant_risk_score: 0.246
                     target: 0
                 event_time: 2026-03-30T05:50:00Z

--- Record: txn-00050 ---
                  record_id: txn-00050
                     amount: 1746.4
                       hour: 18
         distance_from_home: 31.18
      transaction_count_24h: 3
           is_international: 0
        merchant_risk_score: 0.773
                     target: 0
                 event_time: 2026-03-30T06:40:00Z

--- Record: txn-00500 ---
                  record_id: txn-00500
                     amount: 598.93
                       hour: 22
         distance_from_home: 71.75
      transaction_count_24h: 4
           is_international: 0
        merchant_risk_score: 0.179
                     target: 0
          

In [7]:
response = featurestore_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    RecordIdentifierValueAsString="txn-00000",
)

feature_vector = {f["FeatureName"]: f["ValueAsString"] for f in response["Record"]}
print("Feature vector ready for inference:")
print(json.dumps(feature_vector, indent=2))

print("\nIn production, this vector would be sent directly to the inference endpoint.")
print("Same features used for training (offline store) and inference (online store).")
print("No training-serving skew.")

Feature vector ready for inference:
{
  "record_id": "txn-00000",
  "amount": "234.63",
  "hour": "6",
  "distance_from_home": "54.31",
  "transaction_count_24h": "7",
  "is_international": "0",
  "merchant_risk_score": "0.246",
  "target": "0",
  "event_time": "2026-03-30T05:50:00Z"
}

In production, this vector would be sent directly to the inference endpoint.
Same features used for training (offline store) and inference (online store).
No training-serving skew.


## STEP 5 -- Query the Offline Store

The offline store writes feature data as **Parquet files in S3**, partitioned by year/month/day/hour. SageMaker automatically creates a **Glue Data Catalog** table, which you can query with **Athena** using standard SQL.

This is how you build training datasets in production: query the offline store for the time period you need, and the features are guaranteed to match what the online store serves.

**Important:** Offline store data can take 5-15 minutes to appear after ingestion. If the query returns no results, wait and retry.

> **Discussion:** Why is the offline store structured as Parquet files rather than a database? Think about the workloads it serves (batch training, analytics).

In [8]:
fg_description = sm_client.describe_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)

offline_config = fg_description.get("OfflineStoreConfig", {})
s3_uri = offline_config.get("S3StorageConfig", {}).get("ResolvedOutputS3Uri", "Not available yet")

glue_catalog = offline_config.get("DataCatalogConfig", {})
table_name = glue_catalog.get("TableName", "Not available yet")
database_name = glue_catalog.get("Database", "sagemaker_featurestore")
catalog_name = glue_catalog.get("Catalog", "AwsDataCatalog")

print("Offline Store Configuration:")
print(f"  S3 URI:      {s3_uri}")
print(f"  Glue DB:     {database_name}")
print(f"  Glue Table:  {table_name}")
print(f"  Catalog:     {catalog_name}")

Offline Store Configuration:
  S3 URI:      s3://sagemaker-us-west-2-407975137156/fraudshield/feature-store/407975137156/sagemaker/us-west-2/offline-store/fraudshield-transaction-features-1774969881/data
  Glue DB:     sagemaker_featurestore
  Glue Table:  fraudshield_transaction_features_1774969881
  Catalog:     AwsDataCatalog


In [9]:
athena_query = f"""
SELECT record_id, amount, hour, distance_from_home,
       transaction_count_24h, is_international,
       merchant_risk_score, target, event_time
FROM "{database_name}"."{table_name}"
LIMIT 10
"""

print("Athena query to run:")
print(athena_query)

print("\nNote: In production, this query runs as part of a SageMaker Processing Job")
print("inside a pipeline. The results become the training dataset.")
print("\nTo run this manually, use the Athena console or the boto3 Athena client.")
print("The offline store data may take 5-15 minutes to appear after ingestion.")

Athena query to run:

SELECT record_id, amount, hour, distance_from_home,
       transaction_count_24h, is_international,
       merchant_risk_score, target, event_time
FROM "sagemaker_featurestore"."fraudshield_transaction_features_1774969881"
LIMIT 10


Note: In production, this query runs as part of a SageMaker Processing Job
inside a pipeline. The results become the training dataset.

To run this manually, use the Athena console or the boto3 Athena client.
The offline store data may take 5-15 minutes to appear after ingestion.


In [10]:
athena_client = boto3.client("athena")

try:
    query_execution = athena_client.start_query_execution(
        QueryString=athena_query.strip(),
        QueryExecutionContext={"Database": database_name, "Catalog": catalog_name},
        ResultConfiguration={
            "OutputLocation": f"s3://{bucket}/{prefix}/athena-results/"
        },
    )
    query_id = query_execution["QueryExecutionId"]
    print(f"Athena query submitted: {query_id}")

    status = "RUNNING"
    while status in ("RUNNING", "QUEUED"):
        time.sleep(3)
        result = athena_client.get_query_execution(QueryExecutionId=query_id)
        status = result["QueryExecution"]["Status"]["State"]
        print(f"  Query status: {status}")

    if status == "SUCCEEDED":
        results = athena_client.get_query_results(QueryExecutionId=query_id)
        columns = [col["Label"] for col in results["ResultSet"]["ResultSetMetadata"]["ColumnInfo"]]
        rows = results["ResultSet"]["Rows"][1:]  # skip header
        data_rows = [[cell.get("VarCharValue", "") for cell in row["Data"]] for row in rows]
        offline_df = pd.DataFrame(data_rows, columns=columns)
        print(f"\nRetrieved {len(offline_df)} rows from offline store:")
        display(offline_df)
    else:
        reason = result["QueryExecution"]["Status"].get("StateChangeReason", "Unknown")
        print(f"\nQuery failed: {reason}")
        print("The offline store data may not be available yet. Wait 5-15 minutes and retry.")

except Exception as e:
    print(f"Athena query error: {e}")
    print("\nThis is expected if the Glue catalog table has not been created yet.")
    print("The offline store data takes 5-15 minutes to appear after ingestion.")
    print("You can continue with the rest of the notebook and come back to this cell.")

Athena query submitted: 7cbe3249-12f5-4cbe-ac09-855db7e24510
  Query status: SUCCEEDED

Retrieved 10 rows from offline store:


,record_id,amount,hour,distance_from_home,transaction_count_24h,is_international,merchant_risk_score,target,event_time
0,txn-00206,53.5,4,18.01,6,0,0.858,1,2026-03-30T09:16:00Z
1,txn-00226,1806.15,11,29.15,4,0,0.468,0,2026-03-30T09:36:00Z
2,txn-00230,791.91,0,16.56,5,0,0.506,0,2026-03-30T09:40:00Z
3,txn-00235,640.88,8,240.02,4,0,0.162,0,2026-03-30T09:45:00Z
4,txn-01340,148.99,15,7.5,3,1,0.953,0,2026-03-31T04:10:00Z
5,txn-01342,1013.51,13,18.29,6,0,0.309,1,2026-03-31T04:12:00Z
6,txn-00112,1327.48,20,34.08,7,0,0.156,0,2026-03-30T07:42:00Z
7,txn-01537,571.47,5,40.14,9,0,0.458,0,2026-03-31T07:27:00Z
8,txn-01567,50.21,19,94.5,7,0,0.564,0,2026-03-31T07:57:00Z
9,txn-00423,341.02,17,65.41,6,0,0.295,0,2026-03-30T12:53:00Z


---
# STAGE 2 -- Data Wrangler, Canvas, and Autopilot

Feature Store manages the storage and serving of features. But how do you get from raw data to those features? And what tools exist for team members who may not write Python? SageMaker offers three services at different levels of abstraction:

1. **Data Wrangler** -- visual data preparation and feature engineering
2. **Canvas** -- no-code ML for business analysts
3. **Autopilot** -- automated model selection and tuning via SDK

## Data Wrangler: Visual Data Preparation

Data Wrangler is a **visual interface inside SageMaker Studio** for data preparation and feature engineering. Instead of writing Pandas code, you build a visual flow of transforms.

### How It Works

```
Raw Data --> Import --> Transform --> Analyze --> Export
  (S3)      (connect)   (visual)    (quality)   (Pipeline / Feature Store / Notebook)
```

### Data Sources

| Source | Description |
|--------|------------|
| Amazon S3 | CSV, Parquet, JSON files |
| Amazon Athena | Query results from Glue Data Catalog |
| Amazon Redshift | Data warehouse tables |
| Snowflake | External data warehouse |
| Databricks | External data platform |

### Built-in Transforms (300+)

| Category | Examples |
|----------|----------|
| Formatting | Rename columns, change types, reorder |
| Missing Values | Drop, fill (mean/median/mode/custom), impute |
| Encoding | One-hot, ordinal, label encoding |
| Numeric | Normalize, standardize, min-max scale, log transform |
| Text | Tokenize, vectorize, regex extract |
| Time Series | Lag features, rolling aggregates, datetime extract |
| Custom | Write your own PySpark or Pandas transform |

### Export Options

Data Wrangler generates code under the hood. When you export, it creates:
- A **SageMaker Processing Job** (runs your transforms at scale)
- A **Pipeline step** (integrates into SageMaker Pipelines)
- A **Feature Store ingestion job** (writes features directly to a Feature Group)
- A **Python notebook** (the generated code for manual inspection)

### When to Use Data Wrangler

| Use Data Wrangler When | Use Code Instead When |
|------------------------|----------------------|
| Exploring a new dataset visually | You already know the transforms needed |
| Building transforms iteratively | Transforms require complex logic |
| Generating a data quality report | You need unit-tested, version-controlled code |
| Onboarding a non-engineer to the data | Performance-critical preprocessing |

> **Discussion:** When would you use Data Wrangler instead of writing Pandas code? When would you not?

*Note: Data Wrangler runs inside SageMaker Studio and launches an ml.m5.4xlarge instance. We cover it conceptually here rather than launching a live instance.*

## Canvas: No-Code Machine Learning

Canvas takes the abstraction one level higher than Data Wrangler. A business analyst who has never written code can:

1. Upload a CSV or connect to a data source
2. Select a target column
3. Canvas automatically builds, trains, and evaluates multiple models
4. Review predictions and a model analysis report

### Target Audience

- Business analysts
- Domain experts
- Citizen data scientists
- Product managers who want quick experiments

### Supported Problem Types

| Problem Type | Description | Example |
|-------------|-------------|----------|
| Binary Classification | Two classes | Fraud vs. not fraud |
| Multi-class Classification | Multiple classes | Transaction category |
| Regression | Continuous target | Transaction amount prediction |
| Time-Series Forecasting | Future value prediction | Daily fraud volume |
| Image Classification | Categorize images | Document type detection |
| Text Analysis | NLP tasks | Sentiment on customer complaints |

### How Canvas Works Under the Hood

Canvas uses **Autopilot** internally to train models. The visual interface hides:
- Algorithm selection
- Hyperparameter tuning
- Feature engineering
- Cross-validation

### Limitations

- Less control over model architecture and hyperparameters
- No custom containers or algorithms
- Limited preprocessing options compared to code
- Cannot integrate directly into code-based pipelines

> **Discussion:** A product manager at FraudShield wants to test whether adding `customer_tenure` as a feature improves fraud detection. Should they use Canvas or ask the ML team? What are the trade-offs?

## Canvas Walkthrough: Building a Quick Build Model

Follow these steps in SageMaker Studio to build a no-code fraud detection model with Canvas. This mirrors what a business analyst at FraudShield would do without writing any code.

---

### Step 1 -- Launch Canvas

1. In the SageMaker console, navigate to **Domains** and click your Studio domain.
2. Next to your user profile, click **Launch** and select **Canvas**.
3. Canvas opens in a new browser tab. The first launch may take several minutes while the application provisions.
4. Wait for the Canvas home screen to load before proceeding.

---

### Step 2 -- Import the FraudShield Dataset

1. In Canvas, click **My datasets** in the left sidebar.
2. Click **Import data** and select **Amazon S3** as the source.
3. Navigate to your bucket and select the FraudShield transactions CSV (e.g., `s3://<your-bucket>/<prefix>/ecommerce_transactions.csv`).
4. Click **Import**. Canvas uploads and previews the dataset.
5. Verify that the column names and row count look correct in the preview table.

---

### Step 3 -- Create a New Model

1. Click **My models** in the left sidebar, then click **New model**.
2. Name the model `fraudshield-canvas-model`.
3. Select the problem type: **Predictive analysis** (binary classification).
4. Click **Create**.

---

### Step 4 -- Select the Target Column

1. On the model build page, select your imported dataset.
2. Click **Select dataset**.
3. In the column selector, choose `is_fraud` (or `target`, depending on your CSV) as the **Target column**.
4. Canvas shows a preview of the input features it will use. Review the auto-detected column types.
5. Deselect any columns that should not be features (e.g., row IDs, timestamps used only for ordering).

---

### Step 5 -- Run a Quick Build

1. Click **Quick build** (not Standard build). Quick Build trains a sample model in approximately 2-15 minutes.
2. While the model trains, Canvas shows a progress indicator. Do not close the tab.
3. Once training completes, Canvas displays model accuracy metrics on the **Analyze** tab.

> **Note:** Quick Build uses a data sample and trains fewer candidates than Standard Build. It is designed for rapid prototyping, not production-quality models.

---

### Step 6 -- Review the Model Analysis

1. On the **Analyze** tab, review the overall model accuracy.
2. Examine the **Column impact** chart -- this shows which features Canvas found most predictive of fraud. Compare this to the feature importance you saw in Monday's Random Forest.
3. Look at the confusion matrix or class-level metrics if available.

---

### Step 7 -- Generate Predictions

1. Click the **Predict** tab.
2. Select **Single prediction**.
3. Enter sample values for a transaction -- try a high-amount, international transaction from a new customer.
4. Observe the predicted label (`is_fraud`: 0 or 1) and the confidence score.
5. Try a second prediction with low-risk values (small amount, domestic, established customer) and compare the results.

---

### Step 8 -- Clean Up

1. Return to **My models** and delete the `fraudshield-canvas-model`.
2. Go to **My datasets** and delete the imported dataset.
3. Close the Canvas tab.

> **Key takeaway:** You just built a fraud detection model without writing a single line of code. Canvas used Autopilot under the hood to select an algorithm, tune hyperparameters, and evaluate the model. The trade-off is control -- you cannot customize the preprocessing, choose the algorithm, or integrate this directly into a code-based pipeline. But for a product manager or business analyst who needs a quick feasibility check, this is powerful.

## Canvas vs Autopilot: Detailed Comparison

Canvas and Autopilot solve overlapping problems but serve different audiences with different levels of control.

| Dimension | Canvas | Autopilot |
|-----------|--------|----------|
| **Audience** | Business analysts, no-code users | ML engineers, data scientists |
| **Interface** | Visual point-and-click in Studio | Python SDK or Studio UI |
| **Input** | Upload CSV or connect to data source | S3 path to CSV |
| **Model Control** | Minimal -- select target column, Canvas decides the rest | Moderate -- set objective metric, problem type, max candidates |
| **Algorithm Selection** | Automatic (uses Autopilot under the hood) | Automatic with visibility into candidates |
| **Generated Artifacts** | Predictions, model analysis report | Notebooks, model artifacts, tuning logs |
| **Custom Preprocessing** | Built-in transforms only | Custom preprocessing via generated notebooks |
| **Deployment** | One-click deploy from Canvas UI | SDK deploy with full endpoint configuration |
| **Cost Control** | Limited -- runs until done | Max candidates, max runtime, instance type selection |
| **Best For** | Quick experiments, stakeholder self-service | Production AutoML, baseline benchmarking |

### Decision Framework

```
Who is building the model?
  |
  +-- Business analyst / non-engineer --> Canvas
  |
  +-- ML engineer / data scientist
        |
        +-- Quick baseline needed --> Autopilot
        |
        +-- Full control needed --> Custom training (Script Mode)
```

> **Discussion:** For FraudShield's production model, would you use Canvas, Autopilot, or custom training? Why?

## STEP 6 -- Autopilot AutoML on Fraud Data

Autopilot is SageMaker's programmatic AutoML service. You point it at a dataset, specify the target column, and it automatically:

1. **Analyzes** the dataset (data types, statistics, class balance)
2. **Generates** candidate pipelines (feature engineering + algorithm combinations)
3. **Trains and tunes** each candidate
4. **Ranks** candidates by your chosen objective metric

We use **F1** as the objective metric because accuracy is misleading for imbalanced fraud data. A model that always predicts "not fraud" achieves high accuracy but catches zero fraud.

In [11]:
autopilot_data = fraud_data.drop(columns=["record_id", "event_time"])

autopilot_csv_key = f"{prefix}/autopilot/input/fraud_autopilot.csv"
autopilot_data.to_csv("fraud_autopilot.csv", index=False)
s3.upload_file("fraud_autopilot.csv", bucket, autopilot_csv_key)

print(f"Uploaded autopilot dataset to s3://{bucket}/{autopilot_csv_key}")
print(f"  Shape: {autopilot_data.shape}")
print(f"  Target distribution:\n{autopilot_data['target'].value_counts()}")

Uploaded autopilot dataset to s3://sagemaker-us-west-2-407975137156/fraudshield/autopilot/input/fraud_autopilot.csv
  Shape: (2000, 7)
  Target distribution:
target
0    1515
1     485
Name: count, dtype: int64


In [12]:
from datetime import datetime

timestamp = datetime.utcnow().strftime("%Y%m%d%H%M")
AUTOPILOT_JOB_NAME = f"fs-ap-{timestamp}"

autopilot_config = {
    "AutoMLJobName": AUTOPILOT_JOB_NAME,
    "InputDataConfig": [
        {
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{bucket}/{autopilot_csv_key}",
                }
            },
            "TargetAttributeName": "target",
        }
    ],
    "OutputDataConfig": {
        "S3OutputPath": f"s3://{bucket}/{prefix}/autopilot/output/",
    },
    "AutoMLJobObjective": {
        "MetricName": "F1",
    },
    "ProblemType": "BinaryClassification",
    "AutoMLJobConfig": {
        "CompletionCriteria": {
            "MaxCandidates": 2,
            "MaxRuntimePerTrainingJobInSeconds": 300,
            "MaxAutoMLJobRuntimeInSeconds": 900,
        }
    },
    "RoleArn": role,
}

print("Autopilot job configuration:")
print(json.dumps(autopilot_config, indent=2, default=str))

print(f"\nJob name: {AUTOPILOT_JOB_NAME}")
print(f"Objective metric: F1")
print(f"Max candidates: 2")
print(f"Max runtime per job: 300 seconds")
print(f"Max total runtime: 1800 seconds (30 minutes)")

Autopilot job configuration:
{
  "AutoMLJobName": "fs-ap-202603311653",
  "InputDataConfig": [
    {
      "DataSource": {
        "S3DataSource": {
          "S3DataType": "S3Prefix",
          "S3Uri": "s3://sagemaker-us-west-2-407975137156/fraudshield/autopilot/input/fraud_autopilot.csv"
        }
      },
      "TargetAttributeName": "target"
    }
  ],
  "OutputDataConfig": {
    "S3OutputPath": "s3://sagemaker-us-west-2-407975137156/fraudshield/autopilot/output/"
  },
  "AutoMLJobObjective": {
    "MetricName": "F1"
  },
  "ProblemType": "BinaryClassification",
  "AutoMLJobConfig": {
    "CompletionCriteria": {
      "MaxCandidates": 2,
      "MaxRuntimePerTrainingJobInSeconds": 300,
      "MaxAutoMLJobRuntimeInSeconds": 900
    }
  },
  "RoleArn": "arn:aws:iam::407975137156:role/service-role/AmazonSageMaker-ExecutionRole-20260327T083133"
}

Job name: fs-ap-202603311653
Objective metric: F1
Max candidates: 2
Max runtime per job: 300 seconds
Max total runtime: 1800 seconds (30 min

/tmp/ipykernel_406/1326732210.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d%H%M")


In [13]:
sm_client.create_auto_ml_job(**autopilot_config)
print(f"Autopilot job '{AUTOPILOT_JOB_NAME}' launched.")
print("This job runs in the background. Check status with the cell below.")
print("\nWhile it runs, continue with the conceptual material.")

Autopilot job 'fs-ap-202603311653' launched.
This job runs in the background. Check status with the cell below.

While it runs, continue with the conceptual material.


In [16]:
response = sm_client.describe_auto_ml_job(AutoMLJobName=AUTOPILOT_JOB_NAME)

status = response["AutoMLJobStatus"]
secondary_status = response.get("AutoMLJobSecondaryStatus", "")
print(f"Autopilot job status: {status}")
print(f"Secondary status:     {secondary_status}")

if status == "Completed":
    best = response["BestCandidate"]
    print(f"\nBest candidate: {best['CandidateName']}")
    for metric in best.get("FinalAutoMLJobObjectiveMetric", {}).get("MetricName", []):
        pass
    obj_metric = best.get("FinalAutoMLJobObjectiveMetric", {})
    print(f"  Metric: {obj_metric.get('MetricName', 'N/A')}")
    print(f"  Value:  {obj_metric.get('Value', 'N/A')}")
elif status == "Failed":
    print(f"\nFailure reason: {response.get('FailureReason', 'Unknown')}")
else:
    print("\nJob is still running. Re-run this cell to check progress.")

Autopilot job status: Completed
Secondary status:     MaxAutoMLJobRuntimeReached


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6 print(f"Secondary status:     {secondary_status}")                                          │
│    7                                                                                             │
│    8 if status == "Completed":                                                                   │
│ ❱  9 │   best = response["BestCandidate"]                                                        │
│   10 │   print(f"\nBest candidate: {best['CandidateName']}")                                     │
│   11 │   for metric in best.get("FinalAutoMLJobObjectiveMetric", {}).get("MetricName", []):      │
│   12 │   │   pass                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyError: 'BestCandidate'

---
# STAGE 3 -- Exporting to Pipelines

**Connecting to Monday:** On Monday you learned about SageMaker Pipelines as DAGs with steps for preprocessing, training, evaluating, and registering models. You also used Model Registry to version and approve models. Today we make the connection concrete: the preprocessing step reads from Feature Store, the training step builds the model, and the registration step writes to Model Registry.

```
Feature Store (offline) --> ProcessingStep (Athena query --> training CSV)
                                |
                                v
                          TrainingStep (train model)
                                |
                                v
                          RegisterModel (Model Registry)
```

## STEP 7 -- Pipeline Concepts Review

A SageMaker Pipeline is a **directed acyclic graph (DAG)** where:
- Each **node** is a step (processing, training, evaluation, registration, condition)
- Each **edge** is a data dependency (one step's output is another step's input)
- SageMaker **infers the graph** from how steps reference each other's outputs

### Pipeline Step Types

| Step Type | Purpose | FraudShield Use Case |
|-----------|---------|---------------------|
| `ProcessingStep` | Data preprocessing, feature extraction | Query Feature Store offline store, prepare training data |
| `TrainingStep` | Model training | Train Random Forest on fraud features |
| `EvaluationStep` | Compute metrics | Calculate F1, precision, recall on validation data |
| `ConditionStep` | Branch logic | Only register model if F1 >= 0.85 |
| `RegisterModel` | Model Registry | Register approved model with metadata |
| `CreateModelStep` | Create deployable model | Prepare model for endpoint deployment |

### How Feature Store Fits In

Without Feature Store, the ProcessingStep loads raw data and computes features from scratch. With Feature Store, the ProcessingStep queries the offline store -- features are already computed and consistent with what the online store serves at inference time.

> **Discussion:** What happens if the ProcessingStep fails? Does the TrainingStep still run? Why or why not?

## STEP 8 -- Define a Pipeline

Below we define a three-step pipeline:

1. **ProcessingStep** -- runs a script that would query the Feature Store offline store and produce a training CSV
2. **TrainingStep** -- trains a Random Forest using the same SKLearn estimator from Friday
3. **RegisterModel** -- registers the trained model in the `fraudshield-rf` Model Package Group from Monday

This is the same pipeline structure from Monday's MLOps discussion, but now with concrete code.

In [46]:
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingOutput, ProcessingInput
from sagemaker.sklearn import SKLearn
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput

pipeline_name = "fs-feature-store-pipeline"

param_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m4.xlarge",
)
param_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval",
)

print("Pipeline parameters defined:")
print(f"  TrainingInstanceType: {param_instance_type.default_value}")
print(f"  ModelApprovalStatus:  {param_approval_status.default_value}")

Pipeline parameters defined:
  TrainingInstanceType: ml.m4.xlarge
  ModelApprovalStatus:  PendingManualApproval


In [47]:
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type="ml.m4.xlarge",
    instance_count=1,
    role=role,
    sagemaker_session=session,
)

processing_step = ProcessingStep(
    name="QueryFeatureStore",
    processor=sklearn_processor,
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/{prefix}/data",
            destination="/opt/ml/processing/input/data",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train/",
            destination=f"s3://{bucket}/{prefix}/pipeline/train/",
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation/",
            destination=f"s3://{bucket}/{prefix}/pipeline/validation/",
        ),
    ],
    code="code/preprocess.py",
)

print("ProcessingStep defined: QueryFeatureStore")
print("  This step would query the Feature Store offline store via Athena,")
print("  split the data into train/validation, and write CSVs to S3.")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


ProcessingStep defined: QueryFeatureStore
  This step would query the Feature Store offline store via Athena,
  split the data into train/validation, and write CSVs to S3.


In [48]:
sklearn_estimator = SKLearn(
    entry_point="train.py",
    source_dir="code/",
    role=role,
    instance_count=1,
    instance_type=param_instance_type,
    framework_version="1.2-1",
    sagemaker_session=session,
    output_path=f"s3://{bucket}/{prefix}/pipeline/output/",
    hyperparameters={"n-estimators": 100, "random-state": 42},
)

training_step = TrainingStep(
    name="TrainFraudModel",
    estimator=sklearn_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=processing_step.properties.ProcessingOutputConfig
            .Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    },
)

print("TrainingStep defined: TrainFraudModel")
print("  Input: output of QueryFeatureStore processing step")
print("  Algorithm: SKLearn RandomForestClassifier")

TrainingStep defined: TrainFraudModel
  Input: output of QueryFeatureStore processing step
  Algorithm: SKLearn RandomForestClassifier


In [49]:
sklearn_image = image_uris.retrieve("sklearn", region, version="1.2-1")

MODEL_PACKAGE_GROUP = "fraudshield-rf"

register_step = RegisterModel(
    name="RegisterFraudModel",
    estimator=sklearn_estimator,
    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.m4.xlarge"],
    transform_instances=["ml.m4.xlarge"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    approval_status=param_approval_status,
)

print("RegisterModel step defined: RegisterFraudModel")
print(f"  Target Model Package Group: {MODEL_PACKAGE_GROUP}")
print(f"  Approval status: PendingManualApproval (parameterized)")
print(f"\n  This is the same registry you created on Monday.")
print(f"  Each pipeline run creates a new model version automatically.")

INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


RegisterModel step defined: RegisterFraudModel
  Target Model Package Group: fraudshield-rf
  Approval status: PendingManualApproval (parameterized)

  This is the same registry you created on Monday.
  Each pipeline run creates a new model version automatically.


In [50]:
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[param_instance_type, param_approval_status],
    steps=[processing_step, training_step, register_step],
    sagemaker_session=session,
)

pipeline_definition = json.loads(pipeline.definition())

print(f"Pipeline '{pipeline_name}' defined with {len(pipeline_definition['Steps'])} steps:")
for step in pipeline_definition["Steps"]:
    print(f"  - {step['Name']} ({step['Type']})")

print(f"\nPipeline DAG:")
print(f"  QueryFeatureStore --> TrainFraudModel --> RegisterFraudModel")
print(f"\nTo create/update this pipeline in SageMaker, run: pipeline.upsert(role_arn=role)")
print(f"To execute it, run: pipeline.start()")

Pipeline 'fs-feature-store-pipeline' defined with 3 steps:
  - QueryFeatureStore (Processing)
  - TrainFraudModel (Training)
  - RegisterFraudModel-RegisterModel (RegisterModel)

Pipeline DAG:
  QueryFeatureStore --> TrainFraudModel --> RegisterFraudModel

To create/update this pipeline in SageMaker, run: pipeline.upsert(role_arn=role)
To execute it, run: pipeline.start()


In [51]:
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-west-2:407975137156:pipeline/fs-feature-store-pipeline',
 'ResponseMetadata': {'RequestId': '74bda2e1-f932-410c-9e94-f329d466bad6',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '74bda2e1-f932-410c-9e94-f329d466bad6',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '115',
   'date': 'Tue, 31 Mar 2026 18:32:18 GMT'},
  'RetryAttempts': 0}}

## STEP 9 -- Connecting to Monday's Model Registry

On Monday you manually created the `fraudshield-rf` Model Package Group and registered a model version with `PendingManualApproval` status. The `RegisterModel` pipeline step above does exactly the same thing, but **automatically**.

### The Complete Lifecycle

```
Feature Store    -->  Pipeline         -->  Model Registry  -->  Endpoint
(features)           (train + eval)        (version + approve)   (deploy)
```

1. **Feature Store** manages the features (today)
2. **Pipeline** automates the train/evaluate/register workflow (today + Monday)
3. **Model Registry** versions and gates deployments (Monday)
4. **Endpoint** serves predictions (Monday)

Every time the pipeline runs successfully:
- A new model version appears in the `fraudshield-rf` group
- It starts with `PendingManualApproval` status
- A human reviewer (or an automated evaluation gate) promotes it to `Approved`
- Only `Approved` versions can be deployed to production endpoints

This is **MLOps in practice** -- the manual steps from Monday are now automated, versioned, and governed.

> **Discussion:** What triggers a pipeline re-run in production? (New data arrives, feature definitions change, scheduled retraining, model drift detected.)

## STEP 10 -- Cleanup

Delete the Feature Group and clean up any resources created today. This removes both the online store (DynamoDB) and offline store (S3/Glue) data.

**Cleanup checklist:**
1. Delete the Feature Group
2. Stop the Autopilot job if still running
3. Verify no endpoints are running
4. Check the billing dashboard

In [44]:
print("--- Cleanup: Feature Group ---")
try:
    sm_client.delete_feature_group(FeatureGroupName=FEATURE_GROUP_NAME)
    print(f"Deleted Feature Group: {FEATURE_GROUP_NAME}")
except Exception as e:
    print(f"Could not delete Feature Group: {e}")

print("\n--- Cleanup: Autopilot Job ---")
try:
    response = sm_client.describe_auto_ml_job(AutoMLJobName=AUTOPILOT_JOB_NAME)
    status = response["AutoMLJobStatus"]
    if status in ("InProgress", "Stopping"):
        sm_client.stop_auto_ml_job(AutoMLJobName=AUTOPILOT_JOB_NAME)
        print(f"Stopped Autopilot job: {AUTOPILOT_JOB_NAME}")
    else:
        print(f"Autopilot job status: {status} (no action needed)")
except Exception as e:
    print(f"Could not check/stop Autopilot job: {e}")

print("\n--- Cleanup: Endpoints ---")
endpoints = sm_client.list_endpoints(
    NameContains="fraudshield",
    StatusEquals="InService",
)["Endpoints"]

if endpoints:
    for ep in endpoints:
        ep_name = ep["EndpointName"]
        sm_client.delete_endpoint(EndpointName=ep_name)
        print(f"Deleted endpoint: {ep_name}")
else:
    print("No active fraudshield endpoints found.")

print("\nCleanup complete. Check the AWS Billing dashboard to verify no unexpected charges.")

--- Cleanup: Feature Group ---
Deleted Feature Group: fraudshield-transaction-features

--- Cleanup: Autopilot Job ---
Autopilot job status: Completed (no action needed)

--- Cleanup: Endpoints ---
No active fraudshield endpoints found.

Cleanup complete. Check the AWS Billing dashboard to verify no unexpected charges.


---
## Wrap-Up and Wednesday Preview

### What You Accomplished Today

1. **Feature Store** -- Created a Feature Group, ingested fraud data, queried the online store for real-time features, and queried the offline store for batch training data. You understand training-serving skew and how Feature Store prevents it.

2. **Data Wrangler, Canvas, and Autopilot** -- Explored SageMaker's higher-level tools for data preparation and automated ML. You know when to use each tool and how they compare.

3. **Pipelines Integration** -- Connected Feature Store to SageMaker Pipelines, building the complete data flow from feature ingestion through model training to Model Registry registration.

### Wednesday Preview

Wednesday moves into **NLP and Transformers**. The question shifts from "how do we manage data and features" to "how do we process and understand text." You will learn tokenization, attention mechanisms, and the Transformer architecture that powers modern language models.

**Reading:** Complete the NLP and Transformer CTs before Wednesday.